# 02 — Modules & npm

Understanding the module system and npm is fundamental — interviewers use this to gauge your day-to-day Node.js experience.

---

## Table of Contents
1. CommonJS Modules (`require` / `module.exports`)
2. ES Modules (`import` / `export`)
3. CommonJS vs ESM
4. Module Resolution Algorithm
5. Module Caching
6. Circular Dependencies
7. npm & package.json
8. Semantic Versioning
9. Interview Questions

---
## 1. CommonJS Modules

CommonJS is the **original** module system in Node.js. Every file is a module.

### Key points:
- `require()` is **synchronous** — it blocks until the module is loaded
- `module.exports` is the object that gets returned when you `require()` a file
- `exports` is a shorthand reference to `module.exports`

In [ ]:
// ===== EXPORTING (how a module exposes functionality) =====

// Method 1: module.exports (most common)
// math.js
// module.exports = { add: (a,b) => a+b, subtract: (a,b) => a-b };

// Method 2: exports shorthand (adds properties to module.exports)
// math.js
// exports.add = (a, b) => a + b;
// exports.subtract = (a, b) => a - b;

// Method 3: Export a single function/class
// logger.js
// module.exports = function log(msg) { console.log(msg); };

// GOTCHA: This BREAKS exports shorthand!
// exports = { add: (a,b) => a+b }; // This reassigns the local variable,
//                                    // does NOT change module.exports!

console.log('exports === module.exports:', exports === module.exports);

In [ ]:
// ===== REQUIRING (how a module consumes another) =====

const fs = require('fs');           // Core module
const path = require('path');       // Core module
// const axios = require('axios');  // npm package (from node_modules)
// const myLib = require('./myLib');// Local file (relative path)

console.log('fs type:', typeof fs);
console.log('path.join example:', path.join('/users', 'john', 'docs'));

In [ ]:
// Destructuring require (common pattern)
const { readFileSync, writeFileSync } = require('fs');
const { join, resolve, basename } = require('path');

console.log('basename of /foo/bar/baz.txt:', basename('/foo/bar/baz.txt'));
console.log('resolve:', resolve('src', 'index.js'));

---
## 2. ES Modules (ESM)

ESM is the **standard** module system for JavaScript (works in both Node.js and browsers).

### How to enable ESM in Node.js:
- Use `.mjs` file extension, OR
- Add `"type": "module"` in `package.json`

### Syntax:
```javascript
// Named exports
export const add = (a, b) => a + b;
export function subtract(a, b) { return a - b; }

// Default export
export default class Calculator { }

// Importing
import Calculator, { add, subtract } from './math.mjs';
import * as math from './math.mjs';

// Dynamic import (works in both CJS and ESM)
const module = await import('./math.mjs');
```

### Key differences from CommonJS:
- `import` is **asynchronous** and statically analyzed
- `import` statements are **hoisted** to the top
- ESM has **strict mode** enabled by default
- No `__filename`, `__dirname` — use `import.meta.url` instead

---
## 3. CommonJS vs ESM — Comparison Table

| Feature | CommonJS | ESM |
|---------|----------|-----|
| Syntax | `require()` / `module.exports` | `import` / `export` |
| Loading | Synchronous | Asynchronous |
| Parsing | Dynamic (runtime) | Static (compile-time) |
| Tree Shaking | Not possible | Supported (dead code elimination) |
| Top-level await | Not supported | Supported |
| `this` at top level | `module.exports` | `undefined` |
| File extension | `.js` (default) | `.mjs` or `"type": "module"` |
| Strict mode | Optional | Always enabled |
| `__filename` / `__dirname` | Available | Use `import.meta.url` |
| Conditional imports | Yes (require in if blocks) | Use dynamic `import()` |

### Interview Tip:
> "CommonJS loads modules synchronously at runtime, while ESM uses static analysis at parse time, enabling tree shaking. For new projects, ESM is preferred, but many existing packages still use CommonJS."

---
## 4. Module Resolution Algorithm

When you call `require('something')`, Node.js follows this order:

### For core modules (like `fs`, `path`, `http`):
1. Returns the built-in module immediately

### For file paths (`./something` or `../something`):
1. Try exact file: `something`
2. Try `something.js`
3. Try `something.json`
4. Try `something.node` (C++ addon)
5. Try `something/index.js`
6. Try `something/index.json`

### For packages (`require('express')`):
1. Check `node_modules` in current directory
2. Walk up parent directories checking each `node_modules`
3. Check global `node_modules`

In [ ]:
// You can see where a module was resolved from:
console.log('fs resolved to:', require.resolve('fs'));
console.log('path resolved to:', require.resolve('path'));

// Module search paths
console.log('Module search paths:', module.paths);

---
## 5. Module Caching

Modules are **cached after the first load**. Subsequent `require()` calls return the cached version.

### Why this matters:
- Singletons are easy — just `module.exports = new MyService()`
- Side effects in a module only run **once**
- Can cause issues if you expect a fresh instance each time

In [ ]:
// Module caching demonstration
const fs1 = require('fs');
const fs2 = require('fs');
console.log('Same reference?', fs1 === fs2); // true — cached!

// View the cache
console.log('Cached modules count:', Object.keys(require.cache).length);

// To bust the cache (rarely needed, mainly for testing):
// delete require.cache[require.resolve('./myModule')];

---
## 6. Circular Dependencies

Node.js handles circular dependencies, but you get a **partially loaded module**.

```javascript
// a.js
console.log('a starting');
exports.done = false;
const b = require('./b.js'); // Goes to load b.js
console.log('in a, b.done =', b.done);
exports.done = true;
console.log('a done');

// b.js  
console.log('b starting');
exports.done = false;
const a = require('./a.js'); // Gets PARTIAL version of a (done = false)
console.log('in b, a.done =', a.done); // false!
exports.done = true;
console.log('b done');

// Output:
// a starting
// b starting
// in b, a.done = false    ← partial load!
// b done
// in a, b.done = true
// a done
```

### Interview Tip:
> Circular dependencies are a code smell. To fix them, extract shared logic into a third module, or use dependency injection.

---
## 7. npm & package.json

### Key `package.json` fields:

```json
{
  "name": "my-app",
  "version": "1.0.0",
  "main": "index.js",          // Entry point for CommonJS
  "module": "index.mjs",       // Entry point for ESM (used by bundlers)
  "type": "module",            // Treat .js files as ESM
  "scripts": {
    "start": "node index.js",
    "dev": "nodemon index.js",
    "test": "jest"
  },
  "dependencies": {},          // Production dependencies
  "devDependencies": {},       // Development only
  "peerDependencies": {},      // Required by consuming package
  "engines": {
    "node": ">=18.0.0"         // Required Node.js version
  }
}
```

### Important npm commands:
```bash
npm init -y                 # Create package.json
npm install express         # Install dependency
npm install -D jest         # Install dev dependency
npm install -g nodemon      # Install globally
npm ci                      # Clean install (uses package-lock.json exactly)
npm audit                   # Check for vulnerabilities
npm outdated                # Check for outdated packages
npm ls                      # List installed packages
npm prune                   # Remove extraneous packages
```

### `npm install` vs `npm ci`:
| Feature | `npm install` | `npm ci` |
|---------|-------------|----------|
| Uses | `package.json` | `package-lock.json` |
| Modifies lock file | Yes | No |
| Deletes `node_modules` | No | Yes (clean) |
| Use case | Development | CI/CD pipelines |

---
## 8. Semantic Versioning (semver)

Format: **MAJOR.MINOR.PATCH** (e.g., `4.17.1`)

| Part | When to increment | Example |
|------|------------------|--------|
| MAJOR | Breaking changes | 3.0.0 → 4.0.0 |
| MINOR | New features (backward compatible) | 4.16.0 → 4.17.0 |
| PATCH | Bug fixes | 4.17.0 → 4.17.1 |

### Version ranges in package.json:
```
"express": "^4.17.1"   →  >=4.17.1 and <5.0.0  (caret: minor + patch updates)
"express": "~4.17.1"   →  >=4.17.1 and <4.18.0  (tilde: patch updates only)
"express": "4.17.1"    →  exactly 4.17.1
"express": "*"         →  any version
"express": ">=4.0.0"   →  4.0.0 or higher
```

### Interview Tip:
> The most common question is "What's the difference between `^` and `~`?" — `^` allows minor updates, `~` allows only patch updates. Use `npm ci` in CI/CD to get deterministic installs from the lock file.

---
## 9. Interview Questions & Answers

### Q1: What's the difference between `module.exports` and `exports`?
**A:** `exports` is a shorthand reference to `module.exports`. They point to the same object initially. But if you reassign `exports = something`, it breaks the reference. Always use `module.exports` when exporting a single value.

### Q2: How does `require()` work internally?
**A:** It resolves the file path, checks the cache, reads the file, wraps it in the module wrapper function, executes it, caches the result, and returns `module.exports`. It's synchronous and blocking.

### Q3: What are circular dependencies and how does Node.js handle them?
**A:** When module A requires module B, and module B requires module A. Node.js handles this by returning a partially loaded version of the first module. This can lead to unexpected `undefined` values. Best practice: refactor to eliminate circular dependencies.

### Q4: What's the difference between `dependencies` and `devDependencies`?
**A:** `dependencies` are packages needed at runtime (Express, Mongoose). `devDependencies` are only needed during development (Jest, ESLint, TypeScript). In production, `npm install --production` skips devDependencies.

### Q5: Explain `package-lock.json`.
**A:** It locks the exact version of every dependency (and transitive dependency) installed. This ensures every developer and CI environment gets the same dependency tree. It should be committed to version control.

### Q6: What is `npx` and when would you use it?
**A:** `npx` executes npm packages without installing them globally. Use it for one-off commands like `npx create-react-app my-app` or running locally installed binaries without adding them to scripts.